# Oral Cancer Model — Exploration & Fine-Tuning Notebook

This notebook lets you:
1. **Test the existing model** `momogueye7/oral_cancer_detection` on your own images
2. **Fine-tune your own model** on the Kaggle OSCC dataset if you want a custom version

**Runtime**: Enable GPU → Runtime > Change runtime type > T4 GPU

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
!pip install -q transformers datasets evaluate accelerate kaggle Pillow

In [ ]:
# ── Cell 2: Test the existing model (no training needed) ─────────────────────
from transformers import pipeline
from PIL import Image
import requests
from io import BytesIO

MODEL_ID = 'momogueye7/oral_cancer_detection'
clf = pipeline('image-classification', model=MODEL_ID)

# Test with a sample image URL (replace with your own image path)
# To test with a local file: img = Image.open('/path/to/your/image.jpg')
# To upload a file in Colab:
from google.colab import files
print('Upload a histopathological image to test:')
uploaded = files.upload()

for filename in uploaded:
    img = Image.open(filename)
    result = clf(img)
    print(f'\nResults for {filename}:')
    for r in result:
        print(f"  {r['label']:20s}  {r['score']:.1%}")

In [ ]:
# ── Cell 3: Check model labels and config ─────────────────────────────────────
from transformers import AutoModelForImageClassification

model = AutoModelForImageClassification.from_pretrained(MODEL_ID)
print('Labels:', model.config.id2label)
print('Architecture:', model.config.model_type)
print('Num labels:', model.config.num_labels)

---
## Optional: Fine-tune your own model on Kaggle OSCC dataset

Run the cells below only if you want to train a custom model under your own HuggingFace account.
Skip these if the existing `momogueye7/oral_cancer_detection` model works well for you.

In [ ]:
# ── Cell 4 (Optional): Mount Drive + Download dataset ─────────────────────────
from google.colab import drive, files
drive.mount('/content/drive')  # saves checkpoints to Drive

print('Upload your kaggle.json (from kaggle.com > Account > Create New Token):')
files.upload()

import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json
!kaggle datasets download -d ashenafifasilkebede/dataset -p /content/data --unzip
!ls /content/data

In [ ]:
# ── Cell 5 (Optional): Fine-tune EfficientNet-B0 ─────────────────────────────
import numpy as np
import evaluate
from torchvision.transforms import (
    RandomResizedCrop, RandomHorizontalFlip, Compose,
    Normalize, ToTensor, Resize, CenterCrop
)
from transformers import (
    AutoImageProcessor, AutoModelForImageClassification,
    TrainingArguments, Trainer, DefaultDataCollator
)
from datasets import load_dataset
from huggingface_hub import notebook_login

notebook_login()  # paste your HF token from huggingface.co/settings/tokens

HF_USERNAME = 'vrkforever'  # your HuggingFace username
MODEL_REPO  = f'{HF_USERNAME}/oral-cancer-binary'
checkpoint  = 'google/efficientnet-b0'

dataset = load_dataset('imagefolder', data_dir='/content/data')
dataset = dataset['train'].train_test_split(test_size=0.2, seed=42)
labels  = dataset['train'].features['label'].names
print('Classes:', labels)

processor = AutoImageProcessor.from_pretrained(checkpoint)
model = AutoModelForImageClassification.from_pretrained(
    checkpoint, num_labels=len(labels),
    id2label={str(i): l for i, l in enumerate(labels)},
    label2id={l: str(i) for i, l in enumerate(labels)},
    ignore_mismatched_sizes=True
)

size = processor.size.get('shortest_edge', 224)
norm = Normalize(mean=processor.image_mean, std=processor.image_std)
train_tf = Compose([RandomResizedCrop(size), RandomHorizontalFlip(), ToTensor(), norm])
val_tf   = Compose([Resize(size), CenterCrop(size), ToTensor(), norm])

def pre_train(b): b['pixel_values'] = [train_tf(i.convert('RGB')) for i in b['image']]; del b['image']; return b
def pre_val(b):   b['pixel_values'] = [val_tf(i.convert('RGB'))   for i in b['image']]; del b['image']; return b

accuracy = evaluate.load('accuracy')
def compute_metrics(ep): return accuracy.compute(predictions=np.argmax(ep[0], axis=1), references=ep[1])

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir='/content/drive/MyDrive/oral-cancer-model',
        remove_unused_columns=False, eval_strategy='epoch', save_strategy='epoch',
        learning_rate=5e-5, per_device_train_batch_size=16, num_train_epochs=5,
        load_best_model_at_end=True, metric_for_best_model='accuracy',
        push_to_hub=True, hub_model_id=MODEL_REPO,
    ),
    data_collator=DefaultDataCollator(),
    train_dataset=dataset['train'].with_transform(pre_train),
    eval_dataset=dataset['test'].with_transform(pre_val),
    processing_class=processor,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.push_to_hub()
print(f'Done! Update MODEL_ID in app.py to: "{MODEL_REPO}"')